# Bootstrap stability analysis ($f_P$, $f_S$)

Stage 4 of the MOF-ECG pipeline. Loads saved `test_predictions.npz` files from the hyperparameter sweep (no retraining) and computes bootstrap means/SDs for accuracy, precision, recall, and specificity.

**Configuration:** set `MOF_CONFIG` (or edit `CONFIG_PATH` below). The sweep grid and diagnosis codes come from your YAML — no hard-coded cohort lists.

**Prerequisites:** `build_datasets.py` and `train_multi_f_array.py` completed.

**Outputs:**
- `hyper_sweep/bootstrap_results.csv`
- `hyper_sweep/bootstrap_results_with_scores.csv` (includes normalised $f_P$, $f_S$)

If those CSVs already exist (e.g. from `python src/mof_analysis.py bootstrap` or a previous run), the notebook **loads them** and skips recomputation. Set `FORCE_REBOOTSTRAP = True` only when you need fresh bootstrap draws.


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Repo src/ on path (notebook may run from notebooks/)
_REPO = Path.cwd().resolve()
if (_REPO / "src" / "mof_analysis.py").is_file():
    SRC = _REPO / "src"
elif (_REPO.parent / "src" / "mof_analysis.py").is_file():
    SRC = _REPO.parent / "src"
else:
    raise FileNotFoundError("Cannot find src/mof_analysis.py — run from repository root or notebooks/")

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from mof_analysis import (
    discover_completed_runs,
    load_analysis_config,
    load_bootstrap_scored,
    pareto_front,
    resolve_paths,
)

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

CONFIG_PATH = os.environ.get("MOF_CONFIG")

cfg = load_analysis_config(CONFIG_PATH)
paths = resolve_paths(cfg)

FORCE_REBOOTSTRAP = False  # True only when you need to recompute from test_predictions.npz
N_BOOTSTRAP = 1000        # used only when FORCE_REBOOTSTRAP=True
F_S_METHOD = "inv_std"     # inv_std | cv | recip_clipped

print(f"Dataset:  {cfg.name}")
print(f"Config:   {cfg.config_path or CONFIG_PATH}")
print(f"Data root: {paths['data_root']}")
print(f"Codes:    {cfg.diagnoses.codes}")
print(f"Expected sweep size: {cfg.sweep_task_count()}")

In [ ]:
if FORCE_REBOOTSTRAP:
    completed, missing, expected = discover_completed_runs(cfg)
    print(f"Completed runs: {len(completed)} / {expected}")
    print(f"Missing runs:   {len(missing)}")
    if missing[:5]:
        print("First missing:", missing[:5])

bs_scored = load_bootstrap_scored(
    cfg,
    force=FORCE_REBOOTSTRAP,
    n_bootstrap=N_BOOTSTRAP,
    f_s_method=F_S_METHOD,
)
bs_scored.head()


In [ ]:
best_perf = (
    bs_scored.sort_values("f_P_raw", ascending=False)
    .groupby("psych_code", as_index=False)
    .first()
    [["psych_code", "architecture", "learning_rate", "dropout_rate", "f_P_raw", "f_S_raw", "f_P", "f_S"]]
)
print("Best config per code (highest f_P_raw):")
display(best_perf)


In [ ]:
# f_P vs f_S scatter with Pareto front (one panel per diagnosis code)
codes = cfg.diagnoses.codes
ncols = min(3, max(1, len(codes)))
nrows = (len(codes) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows), squeeze=False)

for ax, code in zip(axes.flatten(), codes):
    sub = bs_scored[bs_scored["psych_code"] == code]
    if sub.empty:
        ax.set_title(f"{code} (no data)")
        ax.axis("off")
        continue
    mask = pareto_front(sub, "f_P", "f_S")
    ax.scatter(sub.loc[~mask, "f_P"], sub.loc[~mask, "f_S"], c="#B0BEC5", s=35, alpha=0.7, label="dominated")
    ax.scatter(sub.loc[mask, "f_P"], sub.loc[mask, "f_S"], c="#2E7D32", s=55, alpha=0.9, label="Pareto")
    ax.set_title(code)
    ax.set_xlabel("$f_P$")
    ax.set_ylabel("$f_S$")
    ax.grid(True, alpha=0.25)

for ax in axes.flatten()[len(codes):]:
    ax.axis("off")

handles, labels = axes[0, 0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.02))
fig.suptitle("Performance vs stability (bootstrap)", y=1.04)
fig.tight_layout()
# out_plot = paths["analysis_dir"] / "bootstrap_fp_vs_fs.png"
# paths["analysis_dir"].mkdir(parents=True, exist_ok=True)
# fig.savefig(out_plot, dpi=150, bbox_inches="tight")
# print(f"Saved: {out_plot}")
plt.show()
